# Fine-Tuning Transformers for Arabic Sarcasm Detection

This notebook fine-tunes pre-trained Arabic language models (e.g., `aubmindlab/bert-base-arabertv02`, `asafaya/bert-base-arabic`, or `MARBERT`) on the ArSarcasm-v2 dataset.

In [6]:
import os
import sys

REPO = "Arabic-sarcasm"

if not os.path.exists(f"/content/{REPO}"):
    !git clone https://github.com/cad0m/{REPO}.git /content/{REPO}
else:
    !git -C /content/{REPO} pull

%cd /content/{REPO}

if f"/content/{REPO}" not in sys.path:
    sys.path.insert(0, f"/content/{REPO}")

!pip install -q uv
!uv pip install --system emoji tashaphyne

Already up to date.
/content/Arabic-sarcasm
Using Python 3.13.15 environment at: /usr
Checked 2 packages in 171ms


In [7]:
import shutil
import sys
sys.path.append("..")

import torch
import torch.nn.functional as F
from datasets import Dataset
from scipy.special import softmax
from torch import nn
from transformers import (AutoModel, AutoTokenizer, EarlyStoppingCallback,
                          Trainer, TrainingArguments)

from src.data import get_data
from src.evaluate import log_result, metrics, save_predictions, show
from src.preprocess import build_pipeline
from src.train import best_threshold, count_params, set_seed

MODELS = {
    "arabert": "aubmindlab/bert-base-arabertv02-twitter",
    "marbert": "UBC-NLP/MARBERTv2",
}
PREP = "html+urls+mentions"

In [10]:
for name in MODELS.values():
    AutoModel.from_pretrained(name)

tokenizers = {key: AutoTokenizer.from_pretrained(name) for key, name in MODELS.items()}

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
train, val, test, report = get_data()

prep = build_pipeline(["html", "urls", "mentions"])
for frame in (train, val, test):
    frame["text"] = frame["tweet"].apply(prep)
    frame["sent"] = frame["sentiment"].str.upper().map({"NEG": 0, "NEU": 1, "POS": 2})

counts = train["sarcasm"].value_counts().sort_index()
WEIGHTS = torch.tensor(len(train) / (2 * counts.values), dtype=torch.float)
WEIGHTS

tensor([0.6040, 2.9044])

In [12]:
def to_dataset(frame, tokenizer):
    data = Dataset.from_dict({"text": frame["text"].tolist(),
                              "labels": frame["sarcasm"].tolist(),
                              "sentiment": frame["sent"].tolist()})
    return data.map(lambda b: tokenizer(b["text"], truncation=True, max_length=96,
                                        return_token_type_ids=False),
                    batched=True, remove_columns=["text"])


ds = {key: {"train": to_dataset(train, tok), "val": to_dataset(val, tok), "test": to_dataset(test, tok)}
      for key, tok in tokenizers.items()}

Map:   0%|          | 0/10781 [00:00<?, ? examples/s]

Map:   0%|          | 0/1255 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10781 [00:00<?, ? examples/s]

Map:   0%|          | 0/1255 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [13]:
class MultiTask(nn.Module):
    def __init__(self, name, weights, lam):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(name)
        for param in self.encoder.parameters():
            param.data = param.data.contiguous()
        size = self.encoder.config.hidden_size
        self.sarcasm_head = nn.Linear(size, 2)
        self.sentiment_head = nn.Linear(size, 3)
        self.register_buffer("weights", weights)
        self.lam = lam

    def forward(self, input_ids, attention_mask, labels=None, sentiment=None):
        cls = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        logits = self.sarcasm_head(cls)
        if labels is None:
            return {"logits": logits}
        loss = F.cross_entropy(logits, labels, weight=self.weights)
        loss = loss + self.lam * F.cross_entropy(self.sentiment_head(cls), sentiment)
        return {"loss": loss, "logits": logits}

In [14]:
ARGS = dict(
    num_train_epochs=5, learning_rate=2e-5, weight_decay=0.01, warmup_steps=0.1,
    per_device_train_batch_size=32, per_device_eval_batch_size=64,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="f1",
    logging_strategy="epoch", report_to="none",
)


def val_scores(pred):
    proba = softmax(pred.predictions, axis=1)[:, 1]
    return metrics(pred.label_ids, proba, best_threshold(pred.label_ids, proba))


def make_trainer(key, lam, seed):
    set_seed(seed)
    model = MultiTask(MODELS[key], WEIGHTS, lam)
    args = TrainingArguments(output_dir=f"checkpoints/{key}_lam{lam}_seed{seed}", seed=seed, **ARGS)
    return Trainer(model=model, args=args,
                   train_dataset=ds[key]["train"], eval_dataset=ds[key]["val"],
                   processing_class=tokenizers[key], compute_metrics=val_scores,
                   callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])

In [15]:
from pathlib import Path


def predict(trainer, dataset):
    out = trainer.predict(dataset)
    return softmax(out.predictions, axis=1)[:, 1], out.label_ids


def log_run(trainer, result, name, lam, seed, key, final=False):
    seconds = result.metrics["train_runtime"]
    proba, y = predict(trainer, ds[key]["val"])
    threshold = best_threshold(y, proba)
    scores = metrics(y, proba, threshold)
    row = {f"val_{k}": v for k, v in scores.items()}

    if final:
        test_proba, test_y = predict(trainer, ds[key]["test"])
        row |= {f"test_{k}": v for k, v in metrics(test_y, test_proba, threshold).items()}
        save_predictions(name, seed, test["id"], test_y, test_proba, threshold)
        if seed == 42:
            Path("../models").mkdir(exist_ok=True)
            torch.save(trainer.model.state_dict(), f"../models/{name}_seed{seed}.pt")

    log_result(model=name, preprocessing=PREP, seed=seed, lam=lam, threshold=threshold,
               train_time=seconds, trained_params=count_params(trainer.model), **row)
    show(name, scores, threshold, seconds)
    shutil.rmtree(trainer.args.output_dir)

In [28]:
trainer = make_trainer("arabert", lam=0, seed=42)
result = trainer.train()
log_run(trainer, result, "arabert_e1", lam=0, seed=42, key="arabert")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1,Pr Auc,Macro F1,Precision,Recall
1,0.494255,0.437820,0.670807,0.687200,0.796183,0.609023,0.746544
2,0.341277,0.417577,0.669643,0.675448,0.798934,0.649351,0.691244
3,0.220252,0.556934,0.672566,0.668429,0.800326,0.646809,0.700461
4,0.125576,0.785963,0.663900,0.662660,0.792009,0.603774,0.737327
5,0.075074,0.869564,0.665281,0.666010,0.792966,0.606061,0.737327


arabert_e1                   f1=0.6726  pr_auc=0.6684  threshold=0.70  562s


In [29]:
trainer = make_trainer("arabert", lam=0.3, seed=42)
result = trainer.train()
log_run(trainer, result, "arabert_e2_lam0.3", lam=0.3, seed=42, key="arabert")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1,Pr Auc,Macro F1,Precision,Recall
1,0.727792,0.612873,0.664000,0.683969,0.790209,0.586572,0.764977
2,0.510406,0.591166,0.676790,0.667937,0.802036,0.639344,0.718894
3,0.375860,0.710509,0.671082,0.669267,0.799323,0.644068,0.700461
4,0.270801,0.905652,0.672234,0.664043,0.797466,0.614504,0.741935


arabert_e2_lam0.3            f1=0.6768  pr_auc=0.6679  threshold=0.66  431s


In [30]:
trainer = make_trainer("arabert", lam=0.5, seed=42)
result = trainer.train()
log_run(trainer, result, "arabert_e2_lam0.5", lam=0.5, seed=42, key="arabert")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1,Pr Auc,Macro F1,Precision,Recall
1,0.873460,0.726610,0.662309,0.683126,0.793368,0.628099,0.700461
2,0.614853,0.702606,0.677130,0.668952,0.803681,0.659389,0.695853
3,0.462202,0.800838,0.675497,0.673759,0.802017,0.648305,0.705069
4,0.343882,0.975154,0.668161,0.668336,0.798228,0.650655,0.686636


arabert_e2_lam0.5            f1=0.6771  pr_auc=0.6690  threshold=0.69  419s


In [31]:
trainer = make_trainer("arabert", lam=1, seed=42)
result = trainer.train()
log_run(trainer, result, "arabert_e2_lam1", lam=1, seed=42, key="arabert")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1,Pr Auc,Macro F1,Precision,Recall
1,1.230523,1.020039,0.658487,0.678081,0.787927,0.591912,0.741935
2,0.869263,0.986279,0.674419,0.678245,0.803555,0.680751,0.668203
3,0.662876,1.061587,0.681818,0.680326,0.807093,0.672646,0.691244
4,0.501598,1.204383,0.671233,0.678229,0.800867,0.665158,0.677419
5,0.405907,1.238616,0.668151,0.676834,0.797928,0.646552,0.691244


arabert_e2_lam1              f1=0.6818  pr_auc=0.6803  threshold=0.76  547s


In [32]:
trainer = make_trainer("marbert", lam=0, seed=42)
result = trainer.train()
log_run(trainer, result, "marbert_e1", lam=0, seed=42, key="marbert")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss,F1,Pr Auc,Macro F1,Precision,Recall
1,0.485713,0.464678,0.653061,0.673816,0.789556,0.642857,0.663594
2,0.309464,0.503317,0.651584,0.658242,0.788558,0.640000,0.663594
3,0.177649,0.690413,0.647887,0.633617,0.780476,0.575000,0.741935


marbert_e1                   f1=0.6531  pr_auc=0.6738  threshold=0.85  370s


In [33]:
trainer = make_trainer("marbert", lam=0.3, seed=42)
result = trainer.train()
log_run(trainer, result, "marbert_e2_lam0.3", lam=0.3, seed=42, key="marbert")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss,F1,Pr Auc,Macro F1,Precision,Recall
1,0.731571,0.644412,0.649770,0.679460,0.788276,0.649770,0.649770
2,0.485079,0.653108,0.656250,0.670718,0.790783,0.636364,0.677419
3,0.326160,0.808109,0.642706,0.651540,0.779870,0.593750,0.700461
4,0.218038,1.025998,0.642534,0.646435,0.783066,0.631111,0.654378


marbert_e2_lam0.3            f1=0.6562  pr_auc=0.6707  threshold=0.70  486s


In [34]:
trainer = make_trainer("marbert", lam=0.5, seed=42)
result = trainer.train()
log_run(trainer, result, "marbert_e2_lam0.5", lam=0.5, seed=42, key="marbert")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss,F1,Pr Auc,Macro F1,Precision,Recall
1,0.877026,0.748287,0.661224,0.687449,0.789523,0.593407,0.746544
2,0.584163,0.747608,0.663883,0.674560,0.792306,0.606870,0.732719
3,0.407777,0.845762,0.661017,0.658490,0.791254,0.611765,0.718894
4,0.282673,1.024532,0.650655,0.647100,0.786341,0.618257,0.686636


marbert_e2_lam0.5            f1=0.6639  pr_auc=0.6746  threshold=0.60  515s


In [35]:
trainer = make_trainer("marbert", lam=1, seed=42)
result = trainer.train()
log_run(trainer, result, "marbert_e2_lam1", lam=1, seed=42, key="marbert")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss,F1,Pr Auc,Macro F1,Precision,Recall
1,1.239668,1.037329,0.650655,0.668608,0.786341,0.618257,0.686636
2,0.821659,1.009083,0.663793,0.672063,0.793773,0.623482,0.709677
3,0.579265,1.101325,0.666667,0.672177,0.792390,0.593525,0.760369
4,0.400989,1.281114,0.663900,0.665051,0.792009,0.603774,0.737327
5,0.297742,1.382332,0.660377,0.657663,0.784734,0.559105,0.806452


marbert_e2_lam1              f1=0.6667  pr_auc=0.6722  threshold=0.64  639s


In [9]:
import pandas as pd
res = pd.read_csv("/content/Arabic-sarcasm/results/results.csv")
res[["model", "val_f1", "val_pr_auc"]].sort_values("val_f1", ascending=False)

,model,val_f1,val_pr_auc
104,arabert_frozen_bilstm_att,0.688742,0.691897
109,arabert_e2_lam1,0.681818,0.680326
108,arabert_e2_lam0.5,0.677130,0.668952
107,arabert_e2_lam0.3,0.676790,0.667937
114,arabert_e1,0.672566,0.668429
...,...,...,...
27,zeroshot_nli_fr,0.353333,0.274104
39,zeroshot_nli_en,0.352814,0.275327
26,zeroshot_nli_fr,0.351005,0.284253
40,zeroshot_nli_en,0.348964,0.264537


In [16]:
trainer = make_trainer("arabert", lam=0, seed=42)
result = trainer.train()
log_run(trainer, result, "arabert_e1", lam=0, seed=42, key="arabert", final=True)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1,Pr Auc,Macro F1,Precision,Recall
1,0.494255,0.437820,0.670807,0.687200,0.796183,0.609023,0.746544
2,0.341277,0.417577,0.669643,0.675448,0.798934,0.649351,0.691244
3,0.220252,0.556934,0.672566,0.668429,0.800326,0.646809,0.700461
4,0.125576,0.785963,0.663900,0.662660,0.792009,0.603774,0.737327
5,0.075074,0.869564,0.665281,0.666010,0.792966,0.606061,0.737327


arabert_e1                   f1=0.6726  pr_auc=0.6684  threshold=0.70  377s


In [17]:
trainer = make_trainer("arabert", lam=0, seed=43)
result = trainer.train()
log_run(trainer, result, "arabert_e1", lam=0, seed=43, key="arabert", final=True)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1,Pr Auc,Macro F1,Precision,Recall
1,0.497013,0.408147,0.658182,0.681124,0.781132,0.543544,0.834101
2,0.330315,0.440544,0.663636,0.678481,0.796069,0.654709,0.672811
3,0.221744,0.576720,0.658683,0.680082,0.786783,0.580986,0.760369
4,0.123495,0.883241,0.657407,0.679965,0.793093,0.660465,0.654378


arabert_e1                   f1=0.6636  pr_auc=0.6785  threshold=0.75  336s


In [18]:
trainer = make_trainer("arabert", lam=0, seed=44)
result = trainer.train()
log_run(trainer, result, "arabert_e1", lam=0, seed=44, key="arabert", final=True)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1,Pr Auc,Macro F1,Precision,Recall
1,0.485300,0.417384,0.650224,0.658015,0.787321,0.633188,0.668203
2,0.343262,0.438929,0.666667,0.686851,0.790519,0.577703,0.788018
3,0.215690,0.694857,0.653595,0.674259,0.788036,0.619835,0.691244
4,0.133163,0.783169,0.651429,0.660931,0.779619,0.555195,0.788018


arabert_e1                   f1=0.6667  pr_auc=0.6869  threshold=0.48  312s


In [19]:
trainer = make_trainer("arabert", lam=1, seed=42)
result = trainer.train()
log_run(trainer, result, "arabert_e2", lam=1, seed=42, key="arabert", final=True)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1,Pr Auc,Macro F1,Precision,Recall
1,1.230523,1.020039,0.658487,0.678081,0.787927,0.591912,0.741935
2,0.869263,0.986279,0.674419,0.678245,0.803555,0.680751,0.668203
3,0.662876,1.061587,0.681818,0.680326,0.807093,0.672646,0.691244
4,0.501598,1.204383,0.671233,0.678229,0.800867,0.665158,0.677419
5,0.405907,1.238616,0.668151,0.676834,0.797928,0.646552,0.691244


arabert_e2                   f1=0.6818  pr_auc=0.6803  threshold=0.76  416s


In [20]:
trainer = make_trainer("arabert", lam=1, seed=43)
result = trainer.train()
log_run(trainer, result, "arabert_e2", lam=1, seed=43, key="arabert", final=True)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1,Pr Auc,Macro F1,Precision,Recall
1,1.242812,0.983683,0.679842,0.683245,0.799502,0.595156,0.792627
2,0.861519,0.973323,0.683036,0.688428,0.807085,0.662338,0.705069
3,0.657670,1.058980,0.690583,0.706021,0.811861,0.672489,0.709677
4,0.506117,1.255860,0.675615,0.702004,0.802665,0.656522,0.695853
5,0.410038,1.259659,0.672606,0.697817,0.800641,0.650862,0.695853


arabert_e2                   f1=0.6906  pr_auc=0.7060  threshold=0.74  456s


In [21]:
trainer = make_trainer("arabert", lam=1, seed=44)
result = trainer.train()
log_run(trainer, result, "arabert_e2", lam=1, seed=44, key="arabert", final=True)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1,Pr Auc,Macro F1,Precision,Recall
1,1.237200,0.986865,0.657471,0.679813,0.792832,0.655963,0.658986
2,0.858606,0.997265,0.673729,0.709095,0.799082,0.623529,0.732719
3,0.659638,1.093852,0.666667,0.688169,0.792081,0.590747,0.764977
4,0.504303,1.173210,0.667984,0.689817,0.792076,0.584775,0.778802


arabert_e2                   f1=0.6737  pr_auc=0.7091  threshold=0.63  313s


In [23]:
res = pd.read_csv("/content/Arabic-sarcasm/results/results.csv")
final = res[res["model"].isin(["arabert_e1", "arabert_e2"]) & res["test_f1"].notna()]

print(final.groupby("model")["seed"].apply(list))
final.groupby("model")[["val_f1", "test_f1", "test_pr_auc", "test_macro_f1"]].agg(["mean", "std"]).round(4)

model
arabert_e1    [42, 43, 44]
arabert_e2    [42, 43, 44]
Name: seed, dtype: object


val_f1         test_f1         test_pr_auc         test_macro_f1  \
              mean     std    mean     std        mean     std          mean   
model                                                                          
arabert_e1  0.6676  0.0045  0.6029  0.0129      0.6397  0.0178        0.7221   
arabert_e2  0.6820  0.0084  0.6028  0.0027      0.6386  0.0079        0.7267   

                    
               std  
model               
arabert_e1  0.0017  
arabert_e2  0.0025

In [26]:
print(sorted(os.listdir("/content/Arabic-sarcasm/predictions")))
print(sorted(os.listdir("/content/models")))

['arabert_e1_seed42.csv', 'arabert_e1_seed43.csv', 'arabert_e1_seed44.csv', 'arabert_e2_seed42.csv', 'arabert_e2_seed43.csv', 'arabert_e2_seed44.csv']
['arabert_e1_seed42.pt', 'arabert_e2_seed42.pt']


In [28]:
from google.colab import drive, files
from src.evaluate import ROOT

drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/sarcasm_models
!cp ../models/*.pt /content/drive/MyDrive/sarcasm_models/
!cd {ROOT} && zip -r /content/small_outputs.zip results predictions
files.download("/content/small_outputs.zip")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  adding: results/ (stored 0%)
  adding: results/results.csv (deflated 66%)
  adding: predictions/ (stored 0%)
  adding: predictions/arabert_e1_seed43.csv (deflated 62%)
  adding: predictions/arabert_e1_seed42.csv (deflated 63%)
  adding: predictions/arabert_e2_seed43.csv (deflated 63%)
  adding: predictions/arabert_e2_seed44.csv (deflated 62%)
  adding: predictions/arabert_e1_seed44.csv (deflated 63%)
  adding: predictions/arabert_e2_seed42.csv (deflated 63%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>